In [1]:
import os, gc, random
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

CUDA available: True
GPU: Tesla T4


In [2]:
def find_all_combined():
    for root, _, files in os.walk("/kaggle/input"):
        for f in files:
            if f.lower() == "all_combined.csv":
                return os.path.join(root, f)
    return None

path = find_all_combined()
print("Found:", path)

df = pd.read_csv(path)
df = df.dropna(subset=["content","score","app"]).copy()
df["content"] = df["content"].astype(str)
df["score"] = df["score"].astype(int)

def rating_to_label(score):
    if score <= 2: return "negative"
    if score == 3: return "mixed"
    return "positive"

df["label"] = df["score"].apply(rating_to_label)

print(df["label"].value_counts())
print("Total rows:", len(df))

Found: /kaggle/input/datasets/odins0n/top-20-play-store-app-reviews-daily-update/all_combined.csv
label
positive    137615
negative     52409
mixed         9956
Name: count, dtype: int64
Total rows: 199980


In [3]:
N = 120_000
df_small = df.sample(N, random_state=SEED).reset_index(drop=True)

train_df, temp_df = train_test_split(
    df_small, test_size=0.2, random_state=SEED, stratify=df_small["label"]
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=SEED, stratify=temp_df["label"]
)

print("Train/Val/Test:", len(train_df), len(val_df), len(test_df))
print("Train distribution:\n", train_df["label"].value_counts(normalize=True).round(3))

Train/Val/Test: 96000 12000 12000
Train distribution:
 label
positive    0.689
negative    0.262
mixed       0.049
Name: proportion, dtype: float64


In [4]:
pos = train_df[train_df["label"]=="positive"]
neg = train_df[train_df["label"]=="negative"]
mix = train_df[train_df["label"]=="mixed"]

target_mix = int(0.20 * len(train_df))   # 20% mixed
mix_over = mix.sample(target_mix, replace=True, random_state=SEED)

train_bal = pd.concat([pos, neg, mix_over], ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)

print("Balanced train distribution:\n", train_bal["label"].value_counts(normalize=True).round(3))
print("Balanced train size:", len(train_bal))

Balanced train distribution:
 label
positive    0.599
negative    0.227
mixed       0.174
Name: proportion, dtype: float64
Balanced train size: 110469


In [5]:
def make_prompt(text):
    text = str(text).strip().replace("\n", " ")
    # Stronger instruction for mixed (both pros+cons, or neutral)
    return (
        "Task: Classify sentiment of the app review as one label: positive, mixed, or negative.\n"
        "Rules:\n"
        "- positive: mostly praise\n"
        "- negative: mostly complaints\n"
        "- mixed: both praise and complaints, OR neutral/unclear\n"
        f"Review: {text}\n"
        "Answer with only one word (positive/mixed/negative):"
    )

for d in (train_bal, val_df, test_df):
    d["prompt"] = d["content"].apply(make_prompt)
    d["target"] = d["label"]

train_bal[["prompt","target"]].head(2)

,prompt,target
0,Task: Classify sentiment of the app review as ...,positive
1,Task: Classify sentiment of the app review as ...,positive


In [6]:
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    TrainingArguments, Trainer
)
from peft import LoraConfig, get_peft_model, TaskType

MODEL_ID = "google/flan-t5-small"  # keep small for speed; can upgrade later

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
base_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q", "v"],
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

trainable params: 344,064 || all params: 77,305,216 || trainable%: 0.4451


In [7]:
MAX_INP = 192   # smaller = faster + less VRAM; helps OOM
MAX_TGT = 8

train_ds = Dataset.from_pandas(train_bal[["prompt","target"]])
val_ds   = Dataset.from_pandas(val_df[["prompt","target"]])
test_ds  = Dataset.from_pandas(test_df[["prompt","target"]])

def tokenize(batch):
    x = tokenizer(batch["prompt"], truncation=True, max_length=MAX_INP)
    y = tokenizer(text_target=batch["target"], truncation=True, max_length=MAX_TGT)
    x["labels"] = y["input_ids"]
    return x

train_tok = train_ds.map(tokenize, batched=True, remove_columns=train_ds.column_names)
val_tok   = val_ds.map(tokenize, batched=True, remove_columns=val_ds.column_names)

collator = DataCollatorForSeq2Seq(tokenizer, model=model)

Map:   0%|          | 0/110469 [00:00<?, ? examples/s]

Map:   0%|          | 0/12000 [00:00<?, ? examples/s]

In [8]:
args = TrainingArguments(
    output_dir="lora_flan_sentiment",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-4,
    num_train_epochs=1,     # start with 1; if mixed still weak, set to 2
    logging_steps=100,
    save_total_limit=1,
    fp16=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=collator,
)

In [9]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Step,Training Loss
100,1.011000
200,0.844506
300,0.824331
400,0.766833
500,0.758862
600,0.754380
700,0.750768
800,0.732749
900,0.715890
1000,0.707247


/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked t

TrainOutput(global_step=3453, training_loss=0.7156093419547569, metrics={'train_runtime': 909.5515, 'train_samples_per_second': 121.454, 'train_steps_per_second': 3.796, 'total_flos': 6524973116762112.0, 'train_loss': 0.7156093419547569, 'epoch': 1.0})

In [10]:
def normalize_label(txt: str) -> str:
    t = (txt or "").strip().lower()
    if "pos" in t: return "positive"
    if "neg" in t: return "negative"
    if "mix" in t or "neu" in t: return "mixed"
    # fallback: mixed (safer than forcing pos/neg)
    return "mixed"

@torch.no_grad()
def predict_labels_batched(prompts, batch_size=16, max_new_tokens=3):
    model.eval()
    preds_all = []
    for i in range(0, len(prompts), batch_size):
        batch_prompts = prompts[i:i+batch_size]
        inputs = tokenizer(
            batch_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_INP
        ).to(model.device)

        out = model.generate(**inputs, max_new_tokens=max_new_tokens)
        preds = tokenizer.batch_decode(out, skip_special_tokens=True)
        preds_all.extend([normalize_label(p) for p in preds])

        del inputs, out
        torch.cuda.empty_cache()
    return preds_all

# Evaluate on 5000 first (fast + credible)
K = 5000
truth = test_df["target"].tolist()[:K]
preds = predict_labels_batched(test_df["prompt"].tolist()[:K], batch_size=16)

acc = np.mean([p == t for p, t in zip(preds, truth)])
print("Test accuracy on", K, "samples:", round(acc*100, 2), "%")

Test accuracy on 5000 samples: 85.22 %


In [11]:
labels = ["negative","mixed","positive"]

print(classification_report(truth, preds, labels=labels, digits=4, zero_division=0))

cm = confusion_matrix(truth, preds, labels=labels)
cm_df = pd.DataFrame(cm,
                     index=[f"true_{l}" for l in labels],
                     columns=[f"pred_{l}" for l in labels])
cm_df

              precision    recall  f1-score   support

    negative     0.7989    0.7651    0.7816      1345
       mixed     0.2500    0.0861    0.1280       244
    positive     0.8851    0.9414    0.9123      3411

    accuracy                         0.8522      5000
   macro avg     0.6447    0.5975    0.6073      5000
weighted avg     0.8309    0.8522    0.8389      5000



,pred_negative,pred_mixed,pred_positive
true_negative,1029,34,282
true_mixed,88,21,135
true_positive,171,29,3211


In [12]:
model.save_pretrained("lora_adapter")
tokenizer.save_pretrained("lora_adapter")
print("✅ Saved LoRA adapter to /kaggle/working/lora_adapter")

✅ Saved LoRA adapter to /kaggle/working/lora_adapter


In [13]:
import os
import json
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix

os.makedirs("results", exist_ok=True)

labels = ["negative", "mixed", "positive"]

# Metrics
report = classification_report(
    truth,
    preds,
    labels=labels,
    digits=4,
    zero_division=0,
    output_dict=True
)

metrics = {
    "accuracy": float(np.mean([p == t for p, t in zip(preds, truth)])),
    "macro_f1": report["macro avg"]["f1-score"],
    "weighted_f1": report["weighted avg"]["f1-score"],
    "per_class_f1": {
        "negative": report["negative"]["f1-score"],
        "mixed": report["mixed"]["f1-score"],
        "positive": report["positive"]["f1-score"]
    },
    "test_samples": len(truth)
}

with open("results/metrics.json", "w") as f:
    json.dump(metrics, f, indent=4)

# Confusion Matrix
cm = confusion_matrix(truth, preds, labels=labels)
cm_df = pd.DataFrame(cm, index=labels, columns=labels)
cm_df.to_csv("results/confusion_matrix.csv")

# Sample predictions
sample_df = pd.DataFrame({
    "review": test_df["content"].tolist()[:5000],
    "true_label": truth,
    "predicted_label": preds
})
sample_df.to_csv("results/sample_predictions.csv", index=False)

print("✅ Saved:")
print("- results/metrics.json")
print("- results/confusion_matrix.csv")
print("- results/sample_predictions.csv")

✅ Saved:
- results/metrics.json
- results/confusion_matrix.csv
- results/sample_predictions.csv


In [14]:
import shutil

shutil.make_archive(
    '/kaggle/working/my_outputs',
    'zip',
    '/kaggle/working'
)

'/kaggle/working/my_outputs.zip'